# CASMI26 v8: cosine floor + fingerprint fills
CPU-only, offline, torch. Writes `submission.csv`.

In [ ]:
"""Subformula labelling (MIST-CF lite, RDKit-free).
Assign each MS2 peak a subformula of the candidate precursor formula.
RDBE filter, ppm matching, adduct-adjusted masses.
"""
import numpy as np
from itertools import product

# monoisotopic masses
ELEM_MASS = {
    "C": 12.0, "H": 1.00782503223, "N": 14.00307400443, "O": 15.99491461957,
    "P": 30.9737619985, "S": 31.9720711744, "F": 18.99840316273,
    "Cl": 34.968852682, "Br": 78.9183376, "I": 126.9044719,
    "Na": 22.9897692809, "K": 38.9637074864,
}
ELEM_ORDER = ["C", "H", "N", "O", "P", "S", "F", "Cl", "Br", "I"]

ADDUCT_DELTA = {
    "[M+H]+": 1.007276, "[M+Na]+": 22.989218, "[M+K]+": 38.963158,
    "[M+NH4]+": 18.033823, "[M-H]-": -1.007276, "[M+Cl]-": 34.968853,
    "[M+CH2O2-H]-": 44.998201, "[M+C2H4O2-H]-": 59.013851,
    "[M]+": 0.0, "[M-H2O+H]+": -17.003348, "[M-2H2O+H]+": -35.013913,
    "[2M+H]+": None, "[2M+Na]+": None, "[2M-H]-": None, "[M+2H]2+": None,
}


def parse_formula(s):
    """'C9H8N2O2' -> dict. Handles two-letter elements."""
    import re
    out = {}
    for el, n in re.findall(r"([A-Z][a-z]?)(\d*)", s):
        if el not in ELEM_MASS:
            return None
        out[el] = out.get(el, 0) + (int(n) if n else 1)
    return out


def formula_mass(f):
    return sum(ELEM_MASS[e] * n for e, n in f.items())


def rdbe(f):
    """Ring-double-bond equivalents. None if elements unsupported."""
    c = f.get("C", 0); h = f.get("H", 0); n = f.get("N", 0)
    hal = sum(f.get(e, 0) for e in ("F", "Cl", "Br", "I"))
    for e in f:
        if e not in ("C", "H", "N", "O", "P", "S", "F", "Cl", "Br", "I"):
            return None
    return c - (h + hal) / 2 + n / 2 + 1


def enumerate_subformulae(prec_f, max_n=200000):
    """All f ⊆ prec_f with RDBE >= 0, as (counts_tuple, mass). Bounded."""
    keys = [e for e in ELEM_ORDER if e in prec_f]
    counts = [prec_f[e] for e in keys]
    # guard combinatorial explosion (e.g. C30H50...): cap by sampling coarse grid
    total = 1
    for c in counts:
        total *= (c + 1)
    subs = []
    if total <= max_n:
        for combo in product(*[range(c + 1) for c in counts]):
            if all(v == 0 for v in combo):
                continue
            f = dict(zip(keys, combo))
            if (rdbe(f) or -1) < 0:
                continue
            subs.append((combo, sum(ELEM_MASS[e] * n for e, n in zip(keys, combo))))
    else:
        # vectorized random sampling for huge combinatorial spaces
        rng = np.random.default_rng(0)
        k = len(keys)
        cm = np.array(counts)
        draws = rng.integers(0, cm + 1, size=(min(max_n * 3, 600000), k))
        draws = np.unique(draws, axis=0)
        nz = draws[np.any(draws > 0, axis=1)][:max_n]
        idx = {e: i for i, e in enumerate(keys)}
        hal_cols = [idx[e] for e in ("F", "Cl", "Br", "I") if e in idx]
        hal = nz[:, hal_cols].sum(axis=1) if hal_cols else 0
        c = nz[:, idx["C"]] if "C" in idx else 0
        h = nz[:, idx["H"]] if "H" in idx else 0
        n = nz[:, idx["N"]] if "N" in idx else 0
        ok = (c - (h + hal) / 2 + n / 2 + 1) >= 0
        sup = ("C", "H", "N", "O", "P", "S", "F", "Cl", "Br", "I")
        if any(e not in sup for e in keys):
            ok = ok & False
        nz = nz[ok][:max_n]
        mv = np.array([ELEM_MASS[e] for e in keys])
        masses = nz @ mv
        subs = [(tuple(row), float(m)) for row, m in zip(nz.tolist(), masses.tolist())]
    return keys, subs


_SUB_CACHE = {}


def subformula_masses(prec_formula_str, max_n=200000):
    """Cached (keys, masses array) for a precursor formula."""
    hit = _SUB_CACHE.get(prec_formula_str)
    if hit is not None:
        return hit
    prec_f = parse_formula(prec_formula_str)
    if prec_f is None:
        return None
    keys, subs = enumerate_subformulae(prec_f, max_n)
    arr = np.array([m for _, m in subs], dtype=float)
    _SUB_CACHE[prec_formula_str] = (keys, arr)
    return keys, arr


def label_peaks(mzs, intens, prec_formula_str, adduct, ppm=15.0, top_n=20):
    """Greedy: for each top-N peak (by intensity), nearest subformula mass within ppm.
    Returns list of (mz, intensity, subformula_mass or None, ppm_err or None).
    Assumes fragments carry precursor adduct (MIST-CF assumption).
    """
    cached = subformula_masses(prec_formula_str)
    if cached is None:
        return [(m, i, None, None) for m, i in zip(mzs, intens)]
    d = ADDUCT_DELTA.get(adduct)
    if d is None:
        return [(m, i, None, None) for m, i in zip(mzs, intens)]
    mzs = np.asarray(mzs, dtype=float); intens = np.asarray(intens, dtype=float)
    order = np.argsort(-intens)[:top_n]
    _, sub_masses = cached
    out = []
    for idx in order:
        target = mzs[idx] - d  # adduct-adjusted neutral fragment mass
        if len(sub_masses) == 0:
            out.append((mzs[idx], intens[idx], None, None))
            continue
        j = int(np.argmin(np.abs(sub_masses - target)))
        err_ppm = abs(sub_masses[j] - target) / max(target, 1e-9) * 1e6
        if err_ppm <= ppm:
            out.append((mzs[idx], intens[idx], float(sub_masses[j]), float(err_ppm)))
        else:
            out.append((mzs[idx], intens[idx], None, None))
    return out


def explained_intensity(mzs, intens, prec_formula_str, adduct, ppm=15.0, top_n=20):
    """Fraction of top-N intensity explained by subformulae. Core v1 feature."""
    labelled = label_peaks(mzs, intens, prec_formula_str, adduct, ppm, top_n)
    tot = sum(i for _, i, _, _ in labelled)
    exp = sum(i for _, i, m, _ in labelled if m is not None)
    n_hit = sum(1 for _, _, m, _ in labelled if m is not None)
    return (exp / tot if tot > 0 else 0.0), n_hit


In [ ]:
"""v2: memory-based fingerprint prediction (MIST retrieval path, no training).
Query spectra -> cosine neighbors among mass-window train spectra ->
cosine-weighted fingerprint blend -> rank candidates by Tanimoto.
"""
import numpy as np, pandas as pd
from bisect import bisect_left, bisect_right

PROJECT = "."  # unused in kernel


def neutral_mass(prec, adduct):
    if adduct == "[2M+H]+": return (prec - 1.007276) / 2
    if adduct == "[2M+Na]+": return (prec - 22.989218) / 2
    if adduct == "[2M-H]-": return (prec + 1.007276) / 2
    d = ADDUCT_DELTA.get(adduct)
    return prec - d if d is not None else np.nan


def cosine(mz1, it1, mz2, it2, tol=0.02):
    a = np.asarray(it1, dtype=float); b = np.asarray(it2, dtype=float)
    na = float(np.sqrt((a * a).sum())); nb = float(np.sqrt((b * b).sum()))
    if na == 0 or nb == 0: return 0.0
    m1 = np.asarray(mz1, dtype=float); m2 = np.asarray(mz2, dtype=float)
    o1 = np.argsort(m1); o2 = np.argsort(m2)
    m1, a = m1[o1], a[o1]; m2, b = m2[o2], b[o2]
    i = j = 0; num = 0.0
    while i < len(m1) and j < len(m2):
        d = m1[i] - m2[j]
        if abs(d) <= tol: num += a[i] * b[j]; i += 1; j += 1
        elif d < 0: i += 1
        else: j += 1
    return num / (na * nb)


def tanimoto(a, b):
    inter = float(np.logical_and(a, b).sum())
    union = float(np.logical_or(a, b).sum())
    return inter / union if union > 0 else 0.0


def load_fp():
    df = pd.read_parquet(f"{PROJECT}/data/fingerprints.parquet")
    return {s: np.unpackbits(np.asarray(f, dtype=np.uint8)) for s, f in zip(df["smiles"], df["fp"])}


def run(n_query=30, seed=1, k_blend=15, ppm=20, min_n=200):
    rng = np.random.default_rng(seed)
    fp = load_fp()
    train = pd.read_parquet(f"{PROJECT}/data/train.parquet",
        columns=["normalized_smiles", "inchikey14", "adduct", "precursor_mz",
                 "ms2_mzs", "ms2_normalized_intensities"])
    train["neutral"] = [neutral_mass(p, a) for p, a in zip(train["precursor_mz"], train["adduct"])]
    train = train[np.isfinite(train["neutral"].values)]
    groups = np.array(train["inchikey14"].unique())
    held = set(rng.choice(groups, size=min(500, len(groups) // 20), replace=False))
    qpool = train[train["inchikey14"].isin(held)]
    structs = np.array(qpool["normalized_smiles"].unique())
    rng.shuffle(structs)
    queries = list(structs[:n_query])

    db = train[~train["inchikey14"].isin(held)]
    struct = db.groupby("normalized_smiles")["neutral"].median().reset_index()
    truth = qpool.groupby("normalized_smiles")["neutral"].median().reset_index()
    struct = pd.concat([struct, truth]).drop_duplicates("normalized_smiles")
    struct = struct.sort_values("neutral").reset_index(drop=True)
    masses = struct["neutral"].values
    smi = struct["normalized_smiles"].values
    # sampled db spectra per structure for neighbor search
    db_samp = db.groupby("normalized_smiles").head(3)

    hits = {1: 0, 5: 0, 25: 0}
    rr = []
    for qi, qs in enumerate(queries):
        qspec = qpool[qpool["normalized_smiles"] == qs]
        qmass = float(np.median([neutral_mass(p, a) for p, a in zip(qspec["precursor_mz"], qspec["adduct"])]))
        tol = qmass * ppm / 1e6
        lo = bisect_left(masses, qmass - tol); hi = bisect_right(masses, qmass + tol)
        pcur = ppm
        while hi - lo < min_n and pcur < 500:
            pcur *= 2; tol = qmass * pcur / 1e6
            lo = bisect_left(masses, qmass - tol); hi = bisect_right(masses, qmass + tol)
        cands = list(smi[lo:hi][:2000])
        win = db_samp[db_samp["normalized_smiles"].isin(set(cands))]
        # neighbor similarities: best cosine of each db spectrum vs query spectra
        sims = []
        for _, r in qspec.iterrows():
            for _, t in win.iterrows():
                c = cosine(r["ms2_mzs"], r["ms2_normalized_intensities"],
                           t["ms2_mzs"], t["ms2_normalized_intensities"])
                if c > 0.01:
                    sims.append((c, t["normalized_smiles"]))
        sims.sort(reverse=True)
        # blend top-k distinct neighbor fingerprints
        seen, num, den = set(), None, 0.0
        for c, s in sims:
            if s in seen: continue
            seen.add(s)
            f = fp.get(s)
            if f is None: continue
            num = c * f if num is None else num + c * f
            den += c
            if len(seen) >= k_blend: break
        pred = (num / den) if den > 0 else None
        scored = []
        for s in cands:
            f = fp.get(s)
            if f is None or pred is None: t = 0.0
            else: t = tanimoto(pred > 0.3, f)
            scored.append((t, s))
        scored.sort(reverse=True)
        rank = next((i + 1 for i, (_, s) in enumerate(scored) if s == qs), 10**9)
        rr.append(1 / rank if rank <= 25 else 0.0)
        for k in hits:
            if rank <= k: hits[k] += 1
        if (qi + 1) % 10 == 0: print(f"{qi+1}/{n_query} MRR25={np.mean(rr):.3f}", flush=True)
    print(f"n={n_query} hit@1={hits[1]/n_query:.3f} hit@5={hits[5]/n_query:.3f} hit@25={hits[25]/n_query:.3f} MRR@25={np.mean(rr):.3f}")




In [ ]:
import torch
import torch.nn as nn
BIN_W, MZ_MAX = 0.1, 1200.0
N_BINS = int(MZ_MAX / BIN_W)


def bin_spectrum(mzs, intens, out=None):
    v = np.zeros(N_BINS, dtype=np.float32) if out is None else out
    if out is not None: v[:] = 0
    m = np.asarray(mzs, dtype=float); it = np.asarray(intens, dtype=float)
    idx = (m / BIN_W).astype(int)
    ok = (idx >= 0) & (idx < N_BINS)
    np.add.at(v, idx[ok], it[ok])
    n = float(np.sqrt((v * v).sum()))
    if n > 0: v /= n
    return v


ADDUCTS = ["[M+H]+", "[M+Na]+", "[M+K]+", "[M+NH4]+", "[M-H]-", "[M+Cl]-",
           "[M+CH2O2-H]-", "[M+C2H4O2-H]-", "[M]+", "[M-H2O+H]+"]


def meta_vec(adduct, precursor_mz, ce_list):
    a = [1.0 if adduct == x else 0.0 for x in ADDUCTS]
    try: ce = float(np.mean(list(ce_list))) if ce_list is not None else 0.0
    except Exception: ce = 0.0
    return np.array(a + [precursor_mz / 1000.0, ce / 100.0], dtype=np.float32)


class FpMLP(nn.Module):
    def __init__(self, d_in=N_BINS + len(ADDUCTS) + 2, d_h=1024, d_out=2048, p=0.2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_in, d_h), nn.ReLU(), nn.Dropout(p),
            nn.Linear(d_h, d_h // 2), nn.ReLU(), nn.Dropout(p),
            nn.Linear(d_h // 2, d_out))
    def forward(self, x): return self.net(x)


def load_frame(n_cap=None, seed=0):
    cols = ["normalized_smiles", "inchikey14", "molecular_formula", "adduct", "precursor_mz",
            "ms2_mzs", "ms2_normalized_intensities", "collision_energy_ev"]
    tr = pd.read_parquet(f"{PROJECT}/data/train.parquet", columns=cols)
    fp = pd.read_parquet(f"{PROJECT}/data/fingerprints.parquet")
    fmap = {s: np.unpackbits(np.asarray(f, dtype=np.uint8)).astype(np.float32)
            for s, f in zip(fp["smiles"], fp["fp"])}
    tr = tr[tr["normalized_smiles"].isin(fmap)]
    if n_cap: tr = tr.sample(n_cap, random_state=seed).reset_index(drop=True)
    return tr, fmap


def featurize(df):
    X = np.zeros((len(df), N_BINS + len(ADDUCTS) + 2), dtype=np.float32)
    for i, r in enumerate(df.itertuples()):
        X[i, :N_BINS] = bin_spectrum(r.ms2_mzs, r.ms2_normalized_intensities)
        X[i, N_BINS:] = meta_vec(r.adduct, r.precursor_mz, r.collision_energy_ev)
    return X


def main(epochs=5, n_train=200000, batch=512, seed=0):
    device = "mps" if torch.backends.mps.is_available() else "cpu"
    print("device", device, flush=True)
    tr, fmap = load_frame()
    groups = np.array(tr["inchikey14"].unique())
    rng = np.random.default_rng(seed)
    rng.shuffle(groups)
    n_val_g = max(50, len(groups) // 20)
    val_g = set(groups[:n_val_g])
    va = tr[tr["inchikey14"].isin(val_g)].sample(3000, random_state=seed)
    pool = tr[~tr["inchikey14"].isin(val_g)]
    trn = pool.sample(min(n_train, len(pool)), random_state=seed)
    print(f"train {len(trn)} val {len(va)}", flush=True)
    Xtr, Ytr = featurize(trn), np.stack([fmap[s] for s in trn["normalized_smiles"]])
    Xva, Yva = featurize(va), np.stack([fmap[s] for s in va["normalized_smiles"]])
    model = FpMLP().to(device)
    opt = torch.optim.Adam(model.parameters(), lr=3e-4)
    lossf = nn.BCEWithLogitsLoss()
    Xt = torch.from_numpy(Xtr); Yt = torch.from_numpy(Ytr)
    Xv = torch.from_numpy(Xva); Yv = torch.from_numpy(Yva)
    n = len(Xt)
    for ep in range(epochs):
        perm = torch.randperm(n)
        tot = 0.0
        model.train()
        for i in range(0, n, batch):
            idx = perm[i:i + batch]
            opt.zero_grad()
            loss = lossf(model(Xt[idx].to(device)), Yt[idx].to(device))
            loss.backward(); opt.step()
            tot += float(loss) * len(idx)
        model.eval()
        with torch.no_grad():
            pv = torch.sigmoid(model(Xv.to(device))).cpu().numpy()
        tan = (np.logical_and(pv > 0.5, Yva > 0.5).sum(1) /
               np.maximum(np.logical_or(pv > 0.5, Yva > 0.5).sum(1), 1)).mean()
        print(f"ep{ep+1} loss={tot/n:.4f} val_tanimoto={tan:.3f}", flush=True)
    torch.save(model.state_dict(), f"{PROJECT}/v2/fp_mlp.pt")
    print("saved v2/fp_mlp.pt", flush=True)


if __name__ == "__main__":
    main()
def bin_spectrum(mzs, intens, out=None):
    v = np.zeros(N_BINS, dtype=np.float32) if out is None else out
    if out is not None: v[:] = 0
    m = np.asarray(mzs, dtype=float); it = np.asarray(intens, dtype=float)
    idx = (m / BIN_W).astype(int)
    ok = (idx >= 0) & (idx < N_BINS)
    np.add.at(v, idx[ok], it[ok])
    n = float(np.sqrt((v * v).sum()))
    if n > 0: v /= n
    return v


ADDUCTS = ["[M+H]+", "[M+Na]+", "[M+K]+", "[M+NH4]+", "[M-H]-", "[M+Cl]-",
           "[M+CH2O2-H]-", "[M+C2H4O2-H]-", "[M]+", "[M-H2O+H]+"]


def meta_vec(adduct, precursor_mz, ce_list):
    a = [1.0 if adduct == x else 0.0 for x in ADDUCTS]
    try: ce = float(np.mean(list(ce_list))) if ce_list is not None else 0.0
    except Exception: ce = 0.0
    return np.array(a + [precursor_mz / 1000.0, ce / 100.0], dtype=np.float32)


class FpMLP(nn.Module):
    def __init__(self, d_in=N_BINS + len(ADDUCTS) + 2, d_h=1024, d_out=2048, p=0.2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_in, d_h), nn.ReLU(), nn.Dropout(p),
            nn.Linear(d_h, d_h // 2), nn.ReLU(), nn.Dropout(p),
            nn.Linear(d_h // 2, d_out))
    def forward(self, x): return self.net(x)




In [ ]:
"""v8 production: cosine top-5 floor + fingerprint dot-product fills.
Writes submission.csv. Needs v8/fp_trans.pt weights (ships via fp dataset).
"""
import numpy as np, pandas as pd, os, pickle
from bisect import bisect_left, bisect_right

import glob as _glob
_comp = _glob.glob("/kaggle/input/**/test.parquet", recursive=True)
_fp = _glob.glob("/kaggle/input/**/coconut_fp.parquet", recursive=True)
IN = __import__("os").path.dirname(_comp[0]) if _comp else "data"
FP = __import__("os").path.dirname(_fp[0]) if _fp else IN
OUT = "/kaggle/working" if _comp else "."
print("IN=", IN, "FP=", FP, "OUT=", OUT)
TOP_N = 150


def neutral_mass(prec, adduct):
    if adduct == "[2M+H]+": return (prec - 1.007276) / 2
    if adduct == "[2M+Na]+": return (prec - 22.989218) / 2
    if adduct == "[2M-H]-": return (prec + 1.007276) / 2
    d = ADDUCT_DELTA.get(adduct)
    return prec - d if d is not None else np.nan


def topn(mz, it, n=TOP_N):
    mz = np.asarray(mz, dtype=float); it = np.asarray(it, dtype=float)
    if len(mz) <= n: return mz, it
    o = np.argsort(-it)[:n]
    return mz[o], it[o]


def window(masses, smi, qmass, ppm=20, min_n=200, cap=2000):
    tol = qmass * ppm / 1e6
    lo = bisect_left(masses, qmass - tol); hi = bisect_right(masses, qmass + tol)
    pcur = ppm
    while hi - lo < min_n and pcur < 500:
        pcur *= 2; tol = qmass * pcur / 1e6
        lo = bisect_left(masses, qmass - tol); hi = bisect_right(masses, qmass + tol)
    return list(smi[lo:hi][:cap])


def window10(masses, smi, qmass, cap=3000):
    tol = max(qmass * 10 / 1e6, 0.01)
    lo = bisect_left(masses, qmass - tol); hi = bisect_right(masses, qmass + tol)
    return list(smi[lo:hi][:cap])


def main():
    device = "cpu"
    model = FpMLP(d_h=1536).to(device)
    model.load_state_dict(torch.load(f"{FP}/fp_trans.pt", map_location=device))
    model.eval()
    test = pd.read_parquet(f"{IN}/test.parquet")
    test["neutral"] = [neutral_mass(p, a) for p, a in zip(test["precursor_mz"], test["adduct"])]
    mol_neutral = test.groupby("molecule_id")["neutral"].median()
    train = pd.read_parquet(f"{IN}/train.parquet",
        columns=["normalized_smiles", "adduct", "precursor_mz", "ms2_mzs", "ms2_normalized_intensities"])
    train["neutral"] = [neutral_mass(p, a) for p, a in zip(train["precursor_mz"], train["adduct"])]
    train = train[np.isfinite(train["neutral"].values)]
    tstruct = train.groupby("normalized_smiles")["neutral"].median()
    tmass = tstruct.sort_values().values
    tsmi = tstruct.sort_values().index.values
    tsamp = train.groupby("normalized_smiles").head(2)
    cf = pd.read_parquet(f"{FP}/coconut_fp.parquet")
    cfp = {s: np.unpackbits(np.asarray(f, dtype=np.uint8)).astype(np.float32)
           for s, f in zip(cf["canonical_smiles"], cf["fp"])}
    co = cf.sort_values("exact_molecular_weight").reset_index(drop=True)
    cmass = co["exact_molecular_weight"].values
    csmi = co["canonical_smiles"].values
    tf = pd.read_parquet(f"{FP}/fingerprints.parquet")
    scol = "normalized_smiles" if "normalized_smiles" in tf.columns else "smiles"
    tfp = {s: np.unpackbits(np.asarray(f, dtype=np.uint8)).astype(np.float32)
           for s, f in zip(tf[scol], tf["fp"])}

    rows = []
    for mi, (mol, spectra) in enumerate(test.groupby("molecule_id")):
        qmass = float(mol_neutral.loc[mol])
        tcands = window(tmass, tsmi, qmass)
        twin = tsamp[tsamp["normalized_smiles"].isin(set(tcands))]
        qspecs = [topn(np.asarray(mz, dtype=float), np.asarray(it, dtype=float))
                  for mz, it in zip(spectra["ms2_mzs"], spectra["ms2_normalized_intensities"])]
        tscored = []
        for s in tcands:
            best = 0.0
            for tmz, tit in zip(twin[twin["normalized_smiles"] == s]["ms2_mzs"],
                               twin[twin["normalized_smiles"] == s]["ms2_normalized_intensities"]):
                dmz, dit = topn(tmz, tit)
                for qmz, qit in qspecs:
                    c = cosine(qmz, qit, dmz, dit)
                    if c > best: best = c
            tscored.append((best, s))
        tscored.sort(reverse=True)
        top5 = [s for _, s in tscored[:5]]
        # fingerprint channel over +-10ppm train+COCONUT
        feats = []
        for _, r in spectra.iterrows():
            v = np.zeros(N_BINS + len(ADDUCTS) + 2, dtype=np.float32)
            v[:N_BINS] = bin_spectrum(r["ms2_mzs"], r["ms2_normalized_intensities"])
            v[N_BINS:] = meta_vec(r["adduct"], r["precursor_mz"], r["collision_energy_ev"])
            feats.append(v)
        with torch.no_grad():
            Z = model(torch.from_numpy(np.stack(feats))).numpy()
        Zn = Z.mean(axis=0)
        Zn = Zn / (np.linalg.norm(Zn) + 1e-9)
        t10 = window10(tmass, tsmi, qmass)
        c10 = window10(cmass, csmi, qmass)
        scored = []
        for s in t10 + c10:
            f = tfp.get(s, cfp.get(s))
            if f is None: continue
            fn = f / (np.linalg.norm(f) + 1e-9)
            scored.append((float(fn @ Zn), s))
        scored.sort(reverse=True)
        seen = set(top5)
        out = list(top5) + [s for _, s in scored if not (s in seen or seen.add(s))][:20]
        rows.append((mol, ";".join(out[:25])))
        if (mi + 1) % 50 == 0: print(f"done {mi+1}/400", flush=True)
    pd.DataFrame(rows, columns=["molecule_id", "smiles"]).to_csv(f"{OUT}/submission.csv", index=False)
    print("wrote submission.csv")



main()
